# REFERENCE SOLUTION: Test: weeks 1 and 2

| | |
|---|---|
| Test ID | `Q-W1W2` |
| Covers | weeks 1 and 2 |
| Questions | 17 |
| Time | about 60 minutes |

> This is the worked version of the test. Only open it after you have sat the test and read `review()` in your own copy.


## Setup

Run this cell. It builds the study every question works on. The data is synthetic and comes from a fixed seed, so your numbers are the same as mine and the same as the review's.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

QUIZ_ID = "Q-W1W2"
TOPIC = "numpy, pandas and matplotlib, everything from weeks 1 and 2"

rng = np.random.default_rng(1202)

# ---------------------------------------------------------------------------
# A small oral PK study in rats. Two dose groups, four animals each, plasma
# sampled over a day, plus an imaging readout with several cells per animal.
# All synthetic, all from the seed above, so your numbers are my numbers.
# ---------------------------------------------------------------------------

N_ANIMALS = 8
NOMINAL = np.array([0.0, 0.5, 1.0, 2.0, 4.0, 8.0, 24.0])
_group_of = ["low"] * 4 + ["high"] * 4
_dose_of = {"low": 1.0, "high": 3.0}

_ka, _ke = 1.2, 0.15
_cl = rng.lognormal(0.0, 0.22, N_ANIMALS)

_rows = []
for _i in range(N_ANIMALS):
    _g = _group_of[_i]
    _scale = 420.0 * _dose_of[_g] / _cl[_i]
    _true = _scale * (np.exp(-_ke * NOMINAL) - np.exp(-_ka * NOMINAL))
    _obs = _true * rng.lognormal(0.0, 0.09, NOMINAL.size)
    _actual = NOMINAL + np.concatenate([[0.0], rng.normal(0.0, 0.035, NOMINAL.size - 1)])
    _actual = np.round(np.maximum(_actual, 0.0), 3)
    for _j, _t in enumerate(NOMINAL):
        _c = np.nan if _j == 0 else round(float(_obs[_j]), 1)
        _rows.append((_i + 1, _g, float(_t), float(_actual[_j]), _c))

# The sample table from the bioanalytical lab. Pre dose samples are below the
# limit of quantification and come back as nan. Animal 3's 0.5 h sample was
# haemolysed and never assayed.
pk = pd.DataFrame(_rows, columns=["ANIMAL", "GROUP", "NOMINAL_H", "ACTUAL_H", "CONC_NG_ML"])
pk.loc[(pk["ANIMAL"] == 3) & (pk["NOMINAL_H"] == 0.5), "CONC_NG_ML"] = np.nan

# Demographics from the clinical database. Animal 6 was never entered.
_wt = np.round(rng.normal(285, 22, N_ANIMALS), 0)
demo = pd.DataFrame({"ANIMAL": np.arange(1, N_ANIMALS + 1),
                     "SEX": ["M", "M", "F", "F", "M", "F", "M", "F"],
                     "BODYWEIGHT_G": _wt})
demo = demo[demo["ANIMAL"] != 6].reset_index(drop=True)

# One row per animal, saying whether it had the adverse event of interest.
ae = pd.DataFrame({"ANIMAL": np.arange(1, N_ANIMALS + 1),
                   "GROUP": _group_of,
                   "EVENT": ["no", "no", "no", "no", "yes", "yes", "no", "yes"]})

# The imaging readout: several cells per animal, and not the same number in
# each, because that is how imaging works.
_cell_rows = []
_base = {"low": 18.0, "high": 46.0}
for _i in range(N_ANIMALS):
    _g = _group_of[_i]
    _n = int(rng.integers(9, 23))
    _animal_eff = _base[_g] + rng.normal(0, 7.0)
    for _c in range(_n):
        _cell_rows.append((_i + 1, _g, _c + 1,
                           round(float(_animal_eff + rng.normal(0, 9.0)), 2)))
cells = pd.DataFrame(_cell_rows, columns=["ANIMAL", "GROUP", "CELL_ID", "INHIB_PCT"])

# Two plain numpy views of the same plasma data, for section A.
_one = pk[pk["ANIMAL"] == 3].sort_values("NOMINAL_H")
time_single = _one["NOMINAL_H"].to_numpy()          # shape (7,), includes 0 h
conc_single = _one["CONC_NG_ML"].to_numpy()         # shape (7,), two nan
conc_matrix = (pk[pk["NOMINAL_H"] > 0]
               .pivot_table(values="CONC_NG_ML", index="ANIMAL", columns="NOMINAL_H")
               .to_numpy())                         # shape (8, 6), one nan
time_matrix = NOMINAL[1:]                           # shape (6,)

print("pk           %d rows, %d animals, %d nan concentrations"
      % (len(pk), pk["ANIMAL"].nunique(), int(pk["CONC_NG_ML"].isna().sum())))
print("demo         %d rows" % len(demo))
print("ae           %d rows" % len(ae))
print("cells        %d rows, %d animals" % (len(cells), cells["ANIMAL"].nunique()))
print("conc_single  shape %s, for animal 3" % (conc_single.shape,))
print("conc_matrix  shape %s, animals down the rows, times across" % (conc_matrix.shape,))
print("time_matrix  %s" % time_matrix)
print("")
print(pk.head(9).to_string(index=False))
print("")
print(cells.head(4).to_string(index=False))


# Section A. numpy

Five questions on plain arrays. No pandas needed, and none of them wants a loop.

## Question a1

`conc_single` holds animal 3's seven concentrations and `time_single` the seven
nominal times. Two of the concentrations are `nan`.

Work out:

- `n_missing`, how many concentrations are `nan`, as an integer,
- `mask_ok`, a boolean array that is True where the concentration is usable,
- `conc_ok` and `time_ok`, the concentrations and their times, keeping only the
  usable samples.

In [ ]:
n_missing = int(np.isnan(conc_single).sum())
mask_ok = ~np.isnan(conc_single)
conc_ok = conc_single[mask_ok]
time_ok = time_single[mask_ok]

print("missing:", n_missing)
print("kept   :", conc_ok)
print("at     :", time_ok)


## Question a2

From the usable samples only, give `cmax`, the highest concentration, and
`tmax`, the time at which it occurred. Both as plain numbers.

In [ ]:
cmax = float(conc_ok.max())
tmax = float(time_ok[np.argmax(conc_ok)])

print("Cmax %r ng/mL at %r h" % (cmax, tmax))


## Question a3

Take the **last three** usable samples and fit a straight line to the log of
the concentration against time.

Give `lambda_z`, the terminal elimination rate constant in per hour, as a
positive number, and `t_half`, the terminal half-life in hours.

In [ ]:
_slope, _intercept = np.polyfit(time_ok[-3:], np.log(conc_ok[-3:]), 1)
lambda_z = float(-_slope)
t_half = float(np.log(2) / lambda_z)

print("lambda_z %r per h, half-life %r h" % (lambda_z, t_half))


## Question a4

`conc_matrix` has one row per animal and one column per sampling time, and one
`nan` in it.

Give three arrays of six numbers each, one value per sampling time, across all
eight animals:

- `mean_by_time`, the mean concentration,
- `sd_by_time`, the **sample** standard deviation,
- `cv_by_time`, the coefficient of variation as a percentage.

The `nan` has to be ignored rather than propagated.

In [ ]:
mean_by_time = np.nanmean(conc_matrix, axis=0)
sd_by_time = np.nanstd(conc_matrix, axis=0, ddof=1)
cv_by_time = 100.0 * sd_by_time / mean_by_time

print("mean:", np.round(mean_by_time, 2) if mean_by_time is not None else None)
print("sd  :", np.round(sd_by_time, 2) if sd_by_time is not None else None)
print("cv %%:", np.round(cv_by_time, 1) if cv_by_time is not None else None)


## Question a5

Give `norm_matrix`, the same shape as `conc_matrix`, where every animal's
profile has been divided by **that animal's own** highest concentration. Each
row should therefore peak at 1.

One expression, no loop.

In [ ]:
norm_matrix = conc_matrix / np.nanmax(conc_matrix, axis=1, keepdims=True)

print(np.round(norm_matrix, 3) if norm_matrix is not None else None)


# Section B. pandas

Six questions on the sample table and the two lookup tables.

## Question b1

From `pk`, build `post`, the rows that are both **after** dosing and have a
concentration, and give `n_post`, how many rows that is.

In [ ]:
post = pk[(pk["NOMINAL_H"] > 0) & pk["CONC_NG_ML"].notna()].copy()
n_post = int(len(post))

print("rows kept:", n_post)


## Question b2

From `post`, build `by_group`: the mean and standard deviation of
`CONC_NG_ML` for every combination of `GROUP` and `NOMINAL_H`.

The result should have `GROUP` and `NOMINAL_H` as ordinary columns, not as an
index, and the two summary columns must be named exactly `CONC_MEAN` and
`CONC_SD`.

In [ ]:
by_group = (post.groupby(["GROUP", "NOMINAL_H"])
            .agg(CONC_MEAN=("CONC_NG_ML", "mean"),
                 CONC_SD=("CONC_NG_ML", "std"))
            .reset_index())

print(by_group.to_string(index=False) if by_group is not None else None)


## Question b3

Join the body weights onto the sample table. Build `merged`, every row of `pk`
with its animal's `SEX` and `BODYWEIGHT_G` attached, and give
`n_missing_weight`, the number of **animals** that have no weight.

Pass the `validate` argument that states what you expect of the two key
columns, so that a duplicated demographics row would raise rather than
silently multiply your data.

In [ ]:
merged = pk.merge(demo, on="ANIMAL", how="left", validate="many_to_one")
n_missing_weight = int(merged.loc[merged["BODYWEIGHT_G"].isna(), "ANIMAL"].nunique())

print("rows: %s, animals with no weight: %s" % (
    len(merged) if merged is not None else None, n_missing_weight))


## Question b4

Add a column to `pk` called `FRAC_OF_PEAK`: each concentration divided by
**that animal's own** highest concentration, so every animal's profile peaks at
1. Put the resulting Series in `frac_of_peak` as well.

This is the pandas version of question a5, and the whole question is which
groupby method broadcasts a per animal number back onto every one of that
animal's rows.

In [ ]:
frac_of_peak = pk["CONC_NG_ML"] / pk.groupby("ANIMAL")["CONC_NG_ML"].transform("max")
pk["FRAC_OF_PEAK"] = frac_of_peak

print(pk[pk["ANIMAL"] == 3].to_string(index=False) if frac_of_peak is not None else None)


## Question b5

Two parts, both about the difference between the time a sample was meant to be
taken and the time it was.

- `max_dev_min`, the largest absolute difference between `ACTUAL_H` and
  `NOMINAL_H` anywhere in the study, **in minutes**,
- `wide`, a table of concentrations with one row per animal and one column per
  **nominal** time, built with `pivot_table`.

In [ ]:
max_dev_min = float((pk["ACTUAL_H"] - pk["NOMINAL_H"]).abs().max() * 60.0)
wide = pk.pivot_table(values="CONC_NG_ML", index="ANIMAL", columns="NOMINAL_H")

print("worst deviation: %r min" % (max_dev_min,))
print(wide.round(1).to_string() if wide is not None else None)


## Question b6

From `ae`, build `inc`, a table of counts with one row per `GROUP` and one
column per `EVENT` value, using `pd.crosstab`.

Then give `pct_high_event`, the percentage of **high dose** animals that had
the event.

In [ ]:
inc = pd.crosstab(ae["GROUP"], ae["EVENT"])
pct_high_event = 100.0 * float(inc.loc["high", "yes"]) / float(inc.loc["high"].sum())

print(inc.to_string() if inc is not None else None)
print("high dose incidence: %r pct" % (pct_high_event,))


# Section C. matplotlib and nested data

Three questions where the figure is the answer, or the aggregation behind it is.

## Question c1

`cells` holds between 9 and 22 cells for each animal. The unit of replication
is the animal, not the cell.

Build:

- `animal_level`, one row per animal, with columns `ANIMAL`, `GROUP`,
  `MEAN_INHIB`, the mean of that animal's own cells, and `N_CELLS`, how many
  cells it had,
- `group_summary`, one row per group, with columns `GROUP`, `MEAN`, `SD` and
  `N`, computed from the animal means rather than from the cells.

Both with the group keys as ordinary columns.

In [ ]:
animal_level = (cells.groupby(["ANIMAL", "GROUP"])["INHIB_PCT"]
                .agg(MEAN_INHIB="mean", N_CELLS="size")
                .reset_index())
group_summary = (animal_level.groupby("GROUP")
                 .agg(MEAN=("MEAN_INHIB", "mean"),
                      SD=("MEAN_INHIB", "std"),
                      N=("MEAN_INHIB", "size"))
                 .reset_index())

print(animal_level.to_string(index=False) if animal_level is not None else None)
print("")
print(group_summary.to_string(index=False) if group_summary is not None else None)


## Question c2

Plot `group_summary` from the question above: the group mean inhibition with
error bars, into `fig_c2` and `ax_c2` from `plt.subplots()`.

The error bars are the standard deviation of the animal means. Label both axes.

The check reads the means and the error bar lengths back off the axes, so what
matters is what you drew, not how you drew it.

In [ ]:
fig_c2, ax_c2 = plt.subplots(figsize=(5, 4))

_g = group_summary.sort_values("GROUP")
ax_c2.errorbar(_g["GROUP"], _g["MEAN"], yerr=_g["SD"], fmt="o", capsize=5)
ax_c2.set_xlabel("dose group")
ax_c2.set_ylabel("inhibition (pct), mean and SD of animal means")
ax_c2.set_title("Q-W1W2 c2")

plt.show()


## Question c3

Plot the mean concentration against nominal time for both dose groups on the
same axes, into `fig_c3` and `ax_c3`, with a **logarithmic y axis**.

Use `by_group` from question b2. One line per group, markers on the points,
axes labelled, and a legend.

In [ ]:
fig_c3, ax_c3 = plt.subplots(figsize=(6, 4))

for _name in ["low", "high"]:
    _sub = by_group[by_group["GROUP"] == _name].sort_values("NOMINAL_H")
    ax_c3.plot(_sub["NOMINAL_H"], _sub["CONC_MEAN"], "o-", label=_name)
ax_c3.set_yscale("log")
ax_c3.set_xlabel("nominal time (h)")
ax_c3.set_ylabel("mean concentration (ng/mL)")
ax_c3.set_title("Q-W1W2 c3")
ax_c3.legend(title="dose")

plt.show()


# Section D. judgement

Three questions with no code to write. Set each variable to the letter of the right answer, as a string, for example "a".

## Question d1

You are putting error bars on the group mean inhibition from question c1, where
each group has 4 animals and 61 to 72 cells.

Which is the right quantity?

- **a.** `cells.groupby("GROUP")["INHIB_PCT"].std()`, the SD of all the cells
- **b.** `animal_level.groupby("GROUP")["MEAN_INHIB"].std()`, the SD of the
  animal means
- **c.** the SD of the animal means divided by the square root of 4
- **d.** the SD of all the cells divided by the square root of the number of
  cells

Set `q_sd` to the letter.

In [ ]:
q_sd = "b"


## Question d2

`conc_matrix` has shape `(8, 6)`: eight animals down the rows, six sampling
times across the columns.

You want **each animal's own peak concentration**, so eight numbers.

- **a.** `np.nanmax(conc_matrix, axis=0)`
- **b.** `np.nanmax(conc_matrix, axis=1)`
- **c.** `np.nanmax(conc_matrix)`
- **d.** `np.nanmax(conc_matrix, keepdims=True)`

Set `q_axis` to the letter.

In [ ]:
q_axis = "b"


## Question d3

You are joining `pk`, which has seven rows per animal, onto `demo`, which
should have exactly one row per animal.

Which `validate` argument states that, so that a duplicated demographics row
would raise instead of multiplying your sample rows?

- **a.** `validate="one_to_one"`
- **b.** `validate="one_to_many"`
- **c.** `validate="many_to_one"`
- **d.** no `validate` argument, and check `len()` afterwards instead

Set `q_validate` to the letter.

In [ ]:
q_validate = "c"


## Score it

Run this cell. It marks every question, gives a subtotal per section, and points you at what to revise for anything you missed. It does not tell you the answers. Do not edit it.

In [ ]:
# Grading helpers. You do not need to read or edit this cell.
import base64 as _b64, json as _json, datetime as _dt, textwrap as _tw
import numpy as _np
import pandas as _pd

# Expected answers, encoded so that you do not read them by accident.
_EXP = _json.loads(_b64.b64decode("eyJuX21pc3NpbmciOjIsIm1hc2tfb2siOltmYWxzZSxmYWxzZSx0cnVlLHRydWUsdHJ1ZSx0cnVlLHRydWVdLCJjb25jX29rIjpbMTc1LjQsMjMzLjksMTg2LjAsOTMuOSw5LjNdLCJ0aW1lX29rIjpbMS4wLDIuMCw0LjAsOC4wLDI0LjBdLCJuX29rIjo1LCJuX2FsbCI6NywiY21heCI6MjMzLjksInRtYXgiOjIuMCwiYXJnbWF4X29rIjoxLCJsYW1iZGFfeiI6MC4xNDgyODAwMDk1MjA3Njc3MiwidF9oYWxmIjo0LjY3NDU4Mjc4OTY4Mzg5OCwibXJ0Ijo2Ljc0Mzk5NzQwODkwMTg1NiwibGFtYmRhX3pfbG9nMTAiOjAuMDY0Mzk3MTg5OTExNDMxMDgsImxhbWJkYV96X2FsbCI6MC4xMzkwMDY3NTc0MjkwNjM1LCJtZWFuX2J5X3RpbWUiOlszMzIuOTQyODU3MTQyODU3Miw0NzIuNDc1LDU3NC4wMTI1LDQ3Ni45NzUsMjYwLjQzNzUsMjEuMzYyNDk5OTk5OTk5OTk3XSwic2RfYnlfdGltZSI6WzE3NC44MzI2NjE0OTAzNzkyLDI4Ni41NjEzODA4NTkzMTk1NywzNzEuNzkyODg1NjUxOTgzMDYsMjk5LjAzNjk4MTcwNzIzMjAzLDE3MC44NzY1Mjk2NzIzMjU1MiwxMi40NTU3NDMwODE1NzgzMDFdLCJzZF9wb3B1bGF0aW9uIjpbMTYxLjg2MzU5MjEwNDUyMzYsMjY4LjA1MzYyNjg2NDEwMzUsMzQ3Ljc4MDM5OTIzNzQzNTQsMjc5LjcyMzQ4Mjg4MDg2MjIsMTU5Ljg0MDM1NzM2ODY4ODMzLDExLjY1MTI4MDc3NzIzNjQ2N10sImN2X2J5X3RpbWUiOls1Mi41MTEzMTE2OTc5NTk5Myw2MC42NTExMjAzNDY5NjQzLDY0Ljc3MDg2OTIxNDg2NjA3LDYyLjY5NDQ3NzAwNzY0ODYyLDY1LjYxMTMzODQ4NzA5NDAzLDU4LjMwNjU3OTY2ODAwODQ0XSwibWVhbl9ieV9hbmltYWwiOlsxNTMuMjk5OTk5OTk5OTk5OTgsMjE4LjQ1MDAwMDAwMDAwMDAyLDEzOS43LDExNy4yNDk5OTk5OTk5OTk5OSw0NjMuOTAwMDAwMDAwMDAwMDMsNjU3LjI2NjY2NjY2NjY2NjcsNTQ4LjQ4MzMzMzMzMzMzMzIsNTIwLjM4MzMzMzMzMzMzMzNdLCJwZWFrcyI6WzIzNi42LDM0NC45LDIzMy45LDE5Mi4zLDY4MS4yLDExNjEuMSw5MjYuMyw4MTYuOF0sIm5vcm1fbWF0cml4IjpbWzAuNjQyODU3MTQyODU3MTQyOCwxLjAsMC45OTU3NzM0NTczMTE5MTg4LDAuNzczNDU3MzExOTE4ODUwNCwwLjQzNzg2OTgyMjQ4NTIwNzA2LDAuMDM3NjE2MjI5OTIzOTIyMjNdLFswLjU0NzExNTEwNTgyNzc3NjEsMC45MDkyNDkwNTc2OTc4ODM2LDEuMCwwLjkwMjI5MDUxODk5MTAxMTksMC40MDM1OTUyNDQ5OTg1NTAzLDAuMDM3OTgyMDIzNzc1MDA3MjVdLFtudWxsLDAuNzQ5ODkzMTE2NzE2NTQ1NiwxLjAsMC43OTUyMTE2Mjg5MDEyMzk5LDAuNDAxNDUzNjEyNjU0OTgwNzUsMC4wMzk3NjA1ODE0NDUwNjJdLFswLjU2NjMwMjY1MjEwNjA4NDIsMC43MjQ5MDg5OTYzNTk4NTQ0LDEuMCwwLjkwNTg3NjIzNTA0OTQwMTgsMC40MTI4OTY1MTU4NjA2MzQ0LDAuMDQ4MzYxOTM0NDc3Mzc5MV0sWzAuNjI2MjQ3Nzk4MDAzNTIzMSwwLjkzNzc1Njg5OTU4ODk2MDYsMS4wLDAuOTY3ODUwODUxNDM4NjM3NiwwLjUxNjQ0MTU3MzY5MzQ4MjEsMC4wMzc3Mjc1Mzk2MzU5MzY1OF0sWzAuNDQ0NDkyMjkxNzkyMjY2LDAuNzIzODgyNTI1MTkxNjI4NywxLjAsMC44MTcwNzAwMTk4MDg4MDIxLDAuMzc5NjM5OTk2NTU0OTkxLDAuMDMxMzQ5NTgyMjkyNjUzNTJdLFswLjQ5MzI1MjcyNTg5ODczNjksMC44NTYwOTQxMzc5NjgyNjA4LDEuMCwwLjcyMDUwMDkxNzYyOTI3NzgsMC40NDcxNTUzNDkyMzg5MDc1LDAuMDM1NzMzNTYzNjQwMjg5MzI0XSxbMC41ODkyNTA3MzQ1NzM5NDcyLDAuNzg2NjA2MjY4MzY0MzQ4OCwxLjAsMC44Mzk4NjI4Nzk1Mjk4NzI3LDAuNTYzOTA3OTMzMzk4NjI4OSwwLjA0Mjk3MjU3NTkwNTk3NDU0XV0sIm1hdHJpeF9zaGFwZSI6WzgsNl0sIm5fcG9zdCI6NDcsIm5fcGsiOjU2LCJieV9ncm91cF9yb3dzIjoxMiwiYnlfZ3JvdXBfbWVhbiI6WzQ3MC4yMjUsNzI4LjcsODk2LjM0OTk5OTk5OTk5OTksNzQwLjM1LDQxNi44NSwzMi41NzUsMTQ5LjksMjE2LjI1LDI1MS42NzUsMjEzLjYsMTA0LjAyNSwxMC4xNV0sImJ5X2dyb3VwX3NkIjpbMzcuODk0MDA3ODgxNjMzOTIsMTAzLjUxNTE4NDA3MTM4Nzk2LDIwMi45ODM4MTcwODg5NDkyLDEzOS4zNDkwMjIyNDI3MTI2LDQ3LjM1MDQzMTE4NDIzODgsNC43ODAwODAxOTQ1ODUzMDUsMzkuOTQ1NDYyODIxMjAxNTk1LDc2LjMwMTU3MjcyMjk3ODY1LDY1LjI5NTg1ODcyNTI0MTI3LDY1LjI1OTA3MzQ0MzIxMjk4LDI1LjQ3MTM0MDEwMzQ3MDU0LDEuOTc1Njg1NTM1MDMxNjU5XSwiYnlfZ3JvdXBfc2RfcG9wIjpbMzIuODE3MTczNDc2NzAyNzEsODkuNjQ2Nzc5MDgzMjQ0MjUsMTc1Ljc4OTE0MjE1NjE2Mzg4LDEyMC42Nzk3OTMyNTQ3MTE5MSw0MS4wMDY2NzYyODU2OTc2OCw0LjEzOTY3MDg4MDYzNzczNywzMi42MTUzMzM4MTcwODY3LDY2LjA3OTEwMDMyNjgwNTMsNTYuNTQ3ODcyNDE3OTc4NzMsNTYuNTE2MDE1NDI5MjU2ODYsMjIuMDU4ODI3NTk4MDM4ODM1LDEuNzEwOTkzODYzMjI2ODY3XSwiYnlfZ3JvdXBfa2V5cyI6WyJoaWdofDAuNSIsImhpZ2h8MSIsImhpZ2h8MiIsImhpZ2h8NCIsImhpZ2h8OCIsImhpZ2h8MjQiLCJsb3d8MC41IiwibG93fDEiLCJsb3d8MiIsImxvd3w0IiwibG93fDgiLCJsb3d8MjQiXSwibWVyZ2VkX3Jvd3MiOjU2LCJtZXJnZWRfY29scyI6WyJBTklNQUwiLCJHUk9VUCIsIk5PTUlOQUxfSCIsIkFDVFVBTF9IIiwiQ09OQ19OR19NTCIsIkZSQUNfT0ZfUEVBSyIsIlNFWCIsIkJPRFlXRUlHSFRfRyJdLCJuX21pc3Npbmdfd2VpZ2h0IjoxLCJuX21pc3Npbmdfd2VpZ2h0X3Jvd3MiOjcsImZyYWNfbGVuIjo1NiwiZnJhY19ub3RuYSI6NDcsImZyYWNfYW5pbWFsMyI6W251bGwsbnVsbCwwLjc0OTg5MzExNjcxNjU0NTYsMS4wLDAuNzk1MjExNjI4OTAxMjM5OSwwLjQwMTQ1MzYxMjY1NDk4MDc1LDAuMDM5NzYwNTgxNDQ1MDYyXSwiZnJhY19zdW0iOjI5LjE5MDI2NzM1MzUwNzU4LCJuX2dyb3Vwc19hbmltYWwiOjgsIm1heF9kZXZfbWluIjo0LjA3OTk5OTk5OTk5OTk3NywibWF4X2Rldl9oIjowLjA2Nzk5OTk5OTk5OTk5OTYyLCJ3aWRlX3NoYXBlIjpbOCw2XSwid2lkZV9jb2xzIjpbMC41LDEuMCwyLjAsNC4wLDguMCwyNC4wXSwid2lkZV9maXJzdF9yb3ciOlsxNTIuMSwyMzYuNiwyMzUuNiwxODMuMCwxMDMuNiw4LjldLCJpbmNfc2hhcGUiOlsyLDJdLCJpbmNfaW5kZXgiOlsiaGlnaCIsImxvdyJdLCJpbmNfY29scyI6WyJubyIsInllcyJdLCJpbmNfdmFsdWVzIjpbWzEsM10sWzQsMF1dLCJwY3RfaGlnaF9ldmVudCI6NzUuMCwicGN0X2hpZ2hfb2Zfc3R1ZHkiOjM3LjUsImFuaW1hbF9yb3dzIjo4LCJuX2NlbGxzX3RvdGFsIjoxMzMsImFuaW1hbF9pZHMiOlsxLDIsMyw0LDUsNiw3LDhdLCJhbmltYWxfZ3JvdXBzIjpbImxvdyIsImxvdyIsImxvdyIsImxvdyIsImhpZ2giLCJoaWdoIiwiaGlnaCIsImhpZ2giXSwiYW5pbWFsX21lYW4iOlszMi41Mjc0OTk5OTk5OTk5OTYsMTUuMzU4MTgxODE4MTgxODE3LDE2LjI2NzA1ODgyMzUyOTQxNCwxNS44NTMwNzY5MjMwNzY5MjMsNTAuNDQzODg4ODg4ODg4ODksNDMuOTk3NSw0Ni4zMjQyODU3MTQyODU3MTUsNTAuMDI2NDcwNTg4MjM1M10sImFuaW1hbF9tZWRpYW4iOlszMy4wMTUsMTMuMzcwMDAwMDAwMDAwMDAxLDE4LjM4LDE1LjE0LDUwLjIsNDUuMDgsNDcuNjQsNTEuNTVdLCJhbmltYWxfbmNlbGxzIjpbMjAsMjIsMTcsMTMsMTgsMTIsMTQsMTddLCJncm91cF9uYW1lcyI6WyJoaWdoIiwibG93Il0sImdyb3VwX21lYW4iOls0Ny42OTgwMzYyOTc4NTI0NzUsMjAuMDAxNDU0MzkxMTk3MDM2XSwiZ3JvdXBfc2QiOlszLjA4NDUwMjc4Mjk0MDY0MTMsOC4zNTg5NTgxNjE4NTU0NV0sImdyb3VwX24iOls0LDRdLCJncm91cF9zZW0iOlsxLjU0MjI1MTM5MTQ3MDMyMDYsNC4xNzk0NzkwODA5Mjc3MjVdLCJwb29sZWRfbWVhbiI6WzQ4LjExMzkzNDQyNjIyOTUwNSwyMC40MzEzODg4ODg4ODg4OV0sInBvb2xlZF9zZCI6WzcuODY5NTM4NTg4NDgzMzQyLDExLjkyMDY5NjMyNDQ5MDQ0MV0sInBvb2xlZF9uIjpbNjEsNzJdLCJxX3NkIjoiYiIsInFfYXhpcyI6ImIiLCJxX3ZhbGlkYXRlIjoiYyJ9").decode("utf-8"))

CHECK_RESULTS = []


class _Todo(Exception):
    """Raised when a part has not been attempted yet."""


def _need(name):
    g = globals()
    if name not in g:
        raise _Todo("`" + name + "` is not defined yet. Run the starter cell for this part.")
    val = g[name]
    if val is None:
        raise _Todo("`" + name + "` is still None. Fill in the # YOUR CODE blank.")
    return val


def _as_scalar(val):
    try:
        arr = _np.asarray(val, dtype=float)
    except Exception:
        return None
    if arr.size != 1:
        return None
    return float(arr.reshape(-1)[0])


def _as_array(val):
    try:
        return _np.asarray(val, dtype=float)
    except Exception:
        return None


def _num_close(a, b, rtol=1e-4, atol=1e-8):
    a = _as_scalar(a)
    if a is None:
        return False
    return bool(_np.isclose(a, float(b), rtol=rtol, atol=atol, equal_nan=True))


def _arr_close(a, b, rtol=1e-4, atol=1e-8):
    a = _as_array(a)
    if a is None:
        return False
    b = _np.asarray(b, dtype=float)
    if a.shape != b.shape:
        return False
    return bool(_np.allclose(a, b, rtol=rtol, atol=atol, equal_nan=True))


def _cols_of(obj):
    try:
        return [str(c) for c in obj.columns]
    except Exception:
        return None


_META = _json.loads(_b64.b64decode("eyJxdWl6X2lkIjoiUS1XMVcyIiwidGl0bGUiOiJUZXN0OiB3ZWVrcyAxIGFuZCAyIiwic2VjdGlvbnMiOlt7ImtleSI6IkEiLCJ0aXRsZSI6Im51bXB5In0seyJrZXkiOiJCIiwidGl0bGUiOiJwYW5kYXMifSx7ImtleSI6IkMiLCJ0aXRsZSI6Im1hdHBsb3RsaWIgYW5kIG5lc3RlZCBkYXRhIn0seyJrZXkiOiJEIiwidGl0bGUiOiJqdWRnZW1lbnQifV0sIml0ZW1zIjpbeyJpZCI6ImExIiwic2VjdGlvbiI6IkEiLCJ0b3BpYyI6ImZpbmRpbmcgYW5kIG1hc2tpbmcgdGhlIHNhbXBsZXMgeW91IGNhbm5vdCB1c2UiLCJyZXZpc2UiOiJXMUQxIHBhcnQgYSwgYW5kIE1pc3NpbmcgdmFsdWVzIGluIHRoZSByZXZpc2lvbiBzaGVldCJ9LHsiaWQiOiJhMiIsInNlY3Rpb24iOiJBIiwidG9waWMiOiJwZWFrIGFuZCB0aW1lIG9mIHBlYWsiLCJyZXZpc2UiOiJXMUQxIHBhcnQgYiwgYW5kIFJlZHVjdGlvbnMgaW4gdGhlIHJldmlzaW9uIHNoZWV0In0seyJpZCI6ImEzIiwic2VjdGlvbiI6IkEiLCJ0b3BpYyI6InRlcm1pbmFsIHNsb3BlIGFuZCBoYWxmLWxpZmUiLCJyZXZpc2UiOiJXMUQxIHBhcnQgYywgYW5kIEZpdHRpbmcgYSBsaW5lIHRvIGxvZ3MgaW4gdGhlIHJldmlzaW9uIHNoZWV0In0seyJpZCI6ImE0Iiwic2VjdGlvbiI6IkEiLCJ0b3BpYyI6ImF4aXMgcmVkdWN0aW9ucywgYW5kIHRoZSBzYW1wbGUgU0QiLCJyZXZpc2UiOiJXMUQyIHBhcnRzIGEgYW5kIGIsIGFuZCBSZWR1Y3Rpb25zIGluIHRoZSByZXZpc2lvbiBzaGVldCJ9LHsiaWQiOiJhNSIsInNlY3Rpb24iOiJBIiwidG9waWMiOiJicm9hZGNhc3Rpbmcgd2l0aCBrZWVwZGltcyIsInJldmlzZSI6IlcxRDIgcGFydCBjLCBhbmQgQnJvYWRjYXN0aW5nIGluIHRoZSByZXZpc2lvbiBzaGVldCJ9LHsiaWQiOiJiMSIsInNlY3Rpb24iOiJCIiwidG9waWMiOiJmaWx0ZXJpbmcgb24gdHdvIGNvbmRpdGlvbnMiLCJyZXZpc2UiOiJXMUQzIHBhcnQgYSwgYW5kIEZpbHRlcmluZyBpbiB0aGUgcmV2aXNpb24gc2hlZXQifSx7ImlkIjoiYjIiLCJzZWN0aW9uIjoiQiIsInRvcGljIjoiZ3JvdXBieS5hZ2cgd2l0aCBuYW1lZCBvdXRwdXRzIiwicmV2aXNlIjoiVzFEMyBwYXJ0IGIsIGFuZCBHcm91cGluZyBpbiB0aGUgcmV2aXNpb24gc2hlZXQifSx7ImlkIjoiYjMiLCJzZWN0aW9uIjoiQiIsInRvcGljIjoiYSBsZWZ0IG1lcmdlLCB2YWxpZGF0ZWQiLCJyZXZpc2UiOiJXMUQzIHBhcnQgYyBhbmQgVzJEMiwgYW5kIE1lcmdlcyBpbiB0aGUgcmV2aXNpb24gc2hlZXQifSx7ImlkIjoiYjQiLCJzZWN0aW9uIjoiQiIsInRvcGljIjoidHJhbnNmb3JtIGFnYWluc3QgYWdnIiwicmV2aXNlIjoiVzJEMSwgYW5kIFRyYW5zZm9ybSBhZ2FpbnN0IGFnZyBpbiB0aGUgcmV2aXNpb24gc2hlZXQifSx7ImlkIjoiYjUiLCJzZWN0aW9uIjoiQiIsInRvcGljIjoibm9taW5hbCBhZ2FpbnN0IGFjdHVhbCB0aW1lcywgYW5kIHBpdm90aW5nIiwicmV2aXNlIjoiVzJEMywgYW5kIFJlc2hhcGluZyBpbiB0aGUgcmV2aXNpb24gc2hlZXQifSx7ImlkIjoiYjYiLCJzZWN0aW9uIjoiQiIsInRvcGljIjoiYW4gaW5jaWRlbmNlIHRhYmxlLCBhbmQgaXRzIGRlbm9taW5hdG9ycyIsInJldmlzZSI6IlcyRDQsIGFuZCBDcm9zc3RhYnMgaW4gdGhlIHJldmlzaW9uIHNoZWV0In0seyJpZCI6ImMxIiwic2VjdGlvbiI6IkMiLCJ0b3BpYyI6ImFnZ3JlZ2F0aW5nIG5lc3RlZCBkYXRhIHRvIHRoZSBhbmltYWwgZmlyc3QiLCJyZXZpc2UiOiJXMUQ1IHBhcnRzIGEgYW5kIGIsIGFuZCBOZXN0ZWQgZGF0YSBpbiB0aGUgcmV2aXNpb24gc2hlZXQifSx7ImlkIjoiYzIiLCJzZWN0aW9uIjoiQyIsInRvcGljIjoiYSBwbG90IHdpdGggU0QgZXJyb3IgYmFycyIsInJldmlzZSI6IlcxRDUgcGFydCBjLCBhbmQgRXJyb3IgYmFycyBpbiB0aGUgcmV2aXNpb24gc2hlZXQifSx7ImlkIjoiYzMiLCJzZWN0aW9uIjoiQyIsInRvcGljIjoiYSBzZW1pbG9nIGNvbmNlbnRyYXRpb24gdGltZSBwbG90IiwicmV2aXNlIjoiVzJENSwgYW5kIExvZyBzY2FsZXMgaW4gdGhlIHJldmlzaW9uIHNoZWV0In0seyJpZCI6ImQxIiwic2VjdGlvbiI6IkQiLCJ0b3BpYyI6IlNELCBTRU0gYW5kIHdoaWNoIG4iLCJyZXZpc2UiOiJ0aGUgaG91c2UgcnVsZXMsIGFuZCBFcnJvciBiYXJzIGluIHRoZSByZXZpc2lvbiBzaGVldCJ9LHsiaWQiOiJkMiIsInNlY3Rpb24iOiJEIiwidG9waWMiOiJ3aGljaCBheGlzIGlzIHdoaWNoIiwicmV2aXNlIjoiVzFEMiwgYW5kIFJlZHVjdGlvbnMgaW4gdGhlIHJldmlzaW9uIHNoZWV0In0seyJpZCI6ImQzIiwic2VjdGlvbiI6IkQiLCJ0b3BpYyI6IndoYXQgdmFsaWRhdGUgaXMgZm9yIiwicmV2aXNlIjoiVzJEMiwgYW5kIE1lcmdlcyBpbiB0aGUgcmV2aXNpb24gc2hlZXQifV19").decode("utf-8"))
_REVIEW = _json.loads(_b64.b64decode("...").decode("utf-8"))


def check():
    """Score the test. Says what is wrong, never what the answer is."""
    global CHECK_RESULTS
    CHECK_RESULTS = []
    for item in _META["items"]:
        fn = globals().get("_part_" + item["id"])
        if fn is None:
            CHECK_RESULTS.append((item["id"], "FAIL", "no check exists for this question"))
            continue
        try:
            problem = fn()
            status = "FAIL" if problem else "PASS"
            msg = problem or ""
        except _Todo as exc:
            status, msg = "TODO", str(exc)
        except Exception as exc:
            status = "FAIL"
            msg = ("could not check this question, it raised %s: %s"
                   % (type(exc).__name__, exc))
        CHECK_RESULTS.append((item["id"], status, msg))
    _print_score()


def _by_id(item_id):
    for item in _META["items"]:
        if item["id"] == item_id:
            return item
    return {}


def _print_score():
    bar = "=" * 72
    print(bar)
    print("%s  |  %s" % (_META["quiz_id"], _META["title"]))
    print(bar)
    marks = 0
    for section in _META["sections"]:
        rows = [(i, s, m) for i, s, m in CHECK_RESULTS
                if _by_id(i).get("section") == section["key"]]
        got = sum(1 for _, s, _ in rows if s == "PASS")
        marks += got
        print("")
        print("Section %s, %s: %d of %d" % (section["key"], section["title"],
                                            got, len(rows)))
        for item_id, status, msg in rows:
            item = _by_id(item_id)
            print("  Question %s: %-4s  %s" % (item_id, status, item.get("topic", "")))
            if status != "PASS" and msg:
                for line in _tw.wrap(msg, width=62):
                    print("        " + line)
            if status != "PASS" and item.get("revise"):
                print("        revise: %s" % item["revise"])
    total = len(CHECK_RESULTS)
    print("")
    print("-" * 72)
    print("Total: %d of %d" % (marks, total))
    missed = [i for i, s, _ in CHECK_RESULTS if s != "PASS"]
    if missed:
        print("Not yet right: %s" % ", ".join(missed))
    print("Now run review(), whatever you scored.")
    print(bar)
    # The machine readable lines the verifier looks for.
    print("")
    for item_id, status, _ in CHECK_RESULTS:
        print("Question %s: %s" % (item_id, status))


def review():
    """The worked review: every question, the model answer, and why."""
    bar = "=" * 72
    print(bar)
    print("REVIEW  |  %s" % _META["title"])
    print(bar)
    print("")
    print("One way to do each question, and the reason it is that way. The checks")
    print("test the numbers and not the route, so a different correct answer is")
    print("still correct.")
    status_of = {i: s for i, s, _ in CHECK_RESULTS}
    for section in _META["sections"]:
        print("")
        print("=" * 72)
        print("SECTION %s, %s" % (section["key"], section["title"].upper()))
        print("=" * 72)
        for item in _META["items"]:
            if item["section"] != section["key"]:
                continue
            got = status_of.get(item["id"], "not scored")
            print("")
            print("-" * 72)
            print("Question %s  [%s]  %s" % (item["id"], got, item.get("topic", "")))
            print("-" * 72)
            entry = _REVIEW.get(item["id"], {})
            code = entry.get("code", "").strip()
            if code:
                print("")
                for line in code.splitlines():
                    print("    " + line)
            text = entry.get("text", "").strip()
            if text:
                print("")
                for para in text.split("\n\n"):
                    for line in _tw.wrap(" ".join(para.split()), width=70):
                        print(line)
                    print("")
    print(bar)
    if CHECK_RESULTS:
        marks = sum(1 for _, s, _ in CHECK_RESULTS if s == "PASS")
        print("You scored %d of %d. Revise what you missed, then move on."
              % (marks, len(CHECK_RESULTS)))
    else:
        print("check() has not been run, so there is no score to report.")
    print(bar)


def send_to_tutor():
    """Build the block you copy to your tutor, along with your code."""
    lines = []
    lines.append("===== SEND TO TUTOR =====")
    lines.append("Test       : %s, %s" % (_META["quiz_id"], _META["title"]))
    lines.append("Date       : %s" % _dt.date.today().isoformat())
    if not CHECK_RESULTS:
        lines.append("check() has not been run yet. Run the scoring cell above first.")
    else:
        marks = sum(1 for _, s, _ in CHECK_RESULTS if s == "PASS")
        lines.append("Score      : %d of %d" % (marks, len(CHECK_RESULTS)))
        for section in _META["sections"]:
            rows = [(i, s) for i, s, _ in CHECK_RESULTS
                    if _by_id(i).get("section") == section["key"]]
            got = sum(1 for _, s in rows if s == "PASS")
            lines.append("  section %s, %s: %d of %d"
                         % (section["key"], section["title"], got, len(rows)))
        lines.append("Questions not right:")
        any_missed = False
        for item_id, status, msg in CHECK_RESULTS:
            if status == "PASS":
                continue
            any_missed = True
            item = _by_id(item_id)
            lines.append("  %s (%s) %s" % (item_id, status, item.get("topic", "")))
            if msg:
                for line in _tw.wrap(msg, width=56):
                    lines.append("      " + line)
        if not any_missed:
            lines.append("  none")
    lines.append("=========================")
    lines.append("Paste the cells you want to talk about below this block.")
    return "\n".join(lines)


def _yerr_values(ax):
    """The half length of every error bar drawn on the axes."""
    import matplotlib.container as _mcont
    out = []
    for cont in ax.containers:
        eb = cont if isinstance(cont, _mcont.ErrorbarContainer) else None
        if eb is None and getattr(cont, "errorbar", None) is not None:
            eb = cont.errorbar
        if eb is None:
            continue
        lines = getattr(eb, "lines", None)
        if not lines or len(lines) < 3 or not lines[2]:
            continue
        for col in lines[2]:
            try:
                segs = col.get_segments()
            except Exception:
                continue
            for seg in segs:
                seg = _np.asarray(seg, dtype=float)
                if seg.shape[0] >= 2 and seg.shape[1] >= 2:
                    out.append(abs(float(seg[-1, 1]) - float(seg[0, 1])) / 2.0)
    return out


def _plotted_means(ax):
    """The y values of the first error bar or bar container on the axes."""
    import matplotlib.container as _mcont
    for cont in ax.containers:
        if isinstance(cont, _mcont.ErrorbarContainer):
            line = cont.lines[0] if cont.lines else None
            if line is not None:
                try:
                    return [float(v) for v in line.get_ydata()]
                except Exception:
                    return []
        if isinstance(cont, _mcont.BarContainer):
            try:
                return [float(v) for v in cont.datavalues]
            except Exception:
                return []
    return []


def _part_a1():
    n_missing = _need("n_missing")
    mask_ok = _need("mask_ok")
    conc_ok = _need("conc_ok")
    time_ok = _need("time_ok")

    got = _as_scalar(n_missing)
    if got is None:
        return "`n_missing` should be a single whole number."
    if int(got) == _EXP["n_ok"]:
        return "`n_missing` counted the usable samples rather than the missing ones."
    if int(got) != _EXP["n_missing"]:
        return "`n_missing` is not the number of nan concentrations."

    arr = _as_array(mask_ok)
    if arr is None or arr.shape != (_EXP["n_all"],):
        return ("`mask_ok` should be a boolean array with one entry per sample, so %d of "
                "them." % _EXP["n_all"])
    flags = [bool(v) for v in arr]
    if flags == [not v for v in _EXP["mask_ok"]]:
        return ("`mask_ok` is inverted. True should mean the sample IS usable, so the mask "
                "is ~np.isnan(conc_single).")
    if flags != _EXP["mask_ok"]:
        return "`mask_ok` does not flag the usable samples."

    for name, val, want in (("conc_ok", conc_ok, _EXP["conc_ok"]),
                            ("time_ok", time_ok, _EXP["time_ok"])):
        arr = _as_array(val)
        if arr is None:
            return "`%s` should be an array of numbers." % name
        if arr.size == _EXP["n_all"]:
            return ("`%s` still has all %d samples. Index it with your mask."
                    % (name, _EXP["n_all"]))
        if arr.size != _EXP["n_ok"]:
            return ("`%s` has %d values but %d samples are usable."
                    % (name, arr.size, _EXP["n_ok"]))
        if not _arr_close(arr, want, rtol=1e-9):
            return "`%s` does not hold the right values." % name
    return None


def _part_a2():
    cmax = _need("cmax")
    tmax = _need("tmax")
    for name, val in (("cmax", cmax), ("tmax", tmax)):
        if _as_scalar(val) is None:
            return "`%s` should be a single number." % name

    if not _num_close(_as_scalar(cmax), _EXP["cmax"], rtol=1e-9):
        return "`cmax` is not the highest usable concentration."
    got = _as_scalar(tmax)
    if _num_close(got, float(_EXP["argmax_ok"])):
        return ("`tmax` is the position of the peak, not the time. Use it to index the "
                "time array.")
    if not _num_close(got, _EXP["tmax"], rtol=1e-9):
        return "`tmax` is not the time at which the peak occurred."
    return None


def _part_a3():
    lam = _need("lambda_z")
    th = _need("t_half")
    for name, val in (("lambda_z", lam), ("t_half", th)):
        if _as_scalar(val) is None:
            return "`%s` should be a single number." % name

    got = _as_scalar(lam)
    if got < 0:
        return ("`lambda_z` is negative. The fitted slope is negative and lambda_z is "
                "defined as its positive counterpart.")
    if _num_close(got, _EXP["lambda_z_log10"], rtol=1e-4):
        return ("`lambda_z` came from log10. The terminal rate constant is defined on the "
                "natural log, and the two differ by a factor of 2.303.")
    if _num_close(got, _EXP["lambda_z_all"], rtol=1e-4):
        return ("`lambda_z` used every usable sample, including the absorption phase, which "
                "drags the slope shallower. The task asks for the last three.")
    if not _num_close(got, _EXP["lambda_z"], rtol=1e-6):
        return "`lambda_z` is not the negative slope of log concentration on time."

    got = _as_scalar(th)
    if _num_close(got, _EXP["mrt"], rtol=1e-6):
        return "`t_half` is 1 / lambda_z. A half-life carries a factor of ln(2)."
    if not _num_close(got, _EXP["t_half"], rtol=1e-6):
        return "`t_half` is not ln(2) divided by lambda_z."
    return None


def _part_a4():
    mean_by_time = _need("mean_by_time")
    sd_by_time = _need("sd_by_time")
    cv_by_time = _need("cv_by_time")

    n_times = _EXP["matrix_shape"][1]
    for name, val in (("mean_by_time", mean_by_time), ("sd_by_time", sd_by_time),
                      ("cv_by_time", cv_by_time)):
        arr = _as_array(val)
        if arr is None:
            return "`%s` should be an array of numbers." % name
        if arr.shape != (n_times,):
            if arr.shape == (_EXP["matrix_shape"][0],):
                return ("`%s` has one value per ANIMAL, so the reduction collapsed the wrong "
                        "axis. axis=0 goes down the columns and leaves one value per time."
                        % name)
            return ("`%s` should hold one value per sampling time, so %d numbers."
                    % (name, n_times))
        if bool(_np.isnan(arr).any()):
            return ("`%s` contains nan. One missing sample turns a plain mean or std into "
                    "nan for that whole column, which is what np.nanmean and np.nanstd are "
                    "for." % name)

    if not _arr_close(_as_array(mean_by_time), _EXP["mean_by_time"], rtol=1e-9):
        return "`mean_by_time` is not the mean concentration at each sampling time."
    got = _as_array(sd_by_time)
    if _arr_close(got, _EXP["sd_population"], rtol=1e-6):
        return ("`sd_by_time` divides by n. numpy needs ddof=1 for the sample SD, which is "
                "what is reported. Pandas .std() already does this, numpy does not.")
    if not _arr_close(got, _EXP["sd_by_time"], rtol=1e-9):
        return "`sd_by_time` is not the sample standard deviation at each time."
    got = _as_array(cv_by_time)
    if _arr_close(got, [v / 100.0 for v in _EXP["cv_by_time"]], rtol=1e-6):
        return "`cv_by_time` is a fraction. The task asks for a percentage."
    if not _arr_close(got, _EXP["cv_by_time"], rtol=1e-9):
        return "`cv_by_time` is not 100 * sd / mean at each time."
    return None


def _part_a5():
    norm = _need("norm_matrix")
    arr = _as_array(norm)
    if arr is None:
        return "`norm_matrix` should be an array."
    want_shape = tuple(_EXP["matrix_shape"])
    if arr.shape != want_shape:
        if arr.shape == want_shape[::-1]:
            return ("`norm_matrix` is transposed. Dividing by a shape (8,) array lines the "
                    "axes up from the right, which is not what you want here.")
        return ("`norm_matrix` has shape %s and it should have shape %s, the same as "
                "conc_matrix." % (arr.shape, want_shape))
    want = [[_np.nan if v is None else v for v in row] for row in _EXP["norm_matrix"]]
    if not _arr_close(arr, want, rtol=1e-9):
        peak_all = _np.nanmax(_np.asarray(_EXP["norm_matrix"], dtype=float))
        row_max = _np.nanmax(arr, axis=1)
        if _np.nanmax(row_max) > 1.0001 or _np.nanmin(row_max) < 0.9999:
            return ("each row of `norm_matrix` should peak at exactly 1, and yours does not. "
                    "That means the divisor was not each animal's own maximum.")
        return "`norm_matrix` is not conc_matrix divided by each animal's own peak."
    return None


def _part_b1():
    post = _need("post")
    n_post = _need("n_post")
    cols = _cols_of(post)
    if cols is None:
        return "`post` should be a pandas DataFrame."
    if len(post) == _EXP["n_pk"]:
        return ("`post` still has every row of pk. Both conditions have to be applied: "
                "after dosing, and with a concentration.")
    if len(post) != _EXP["n_post"]:
        return ("`post` has %d rows and it should have %d. Pre dose rows and rows with a "
                "nan concentration both go." % (len(post), _EXP["n_post"]))
    if "CONC_NG_ML" not in cols:
        return "`post` should keep all of the original columns."
    if bool(post["CONC_NG_ML"].isna().any()):
        return "`post` still contains nan concentrations."
    got = _as_scalar(n_post)
    if got is None or int(got) != _EXP["n_post"]:
        return "`n_post` should be the number of rows in `post`."
    return None


def _part_b2():
    by_group = _need("by_group")
    cols = _cols_of(by_group)
    if cols is None:
        return "`by_group` should be a pandas DataFrame."
    for want in ("GROUP", "NOMINAL_H"):
        if want not in cols:
            idx = [str(n) for n in (by_group.index.names or [])]
            if want in idx:
                return ("`%s` is in the index rather than a column. Call .reset_index() "
                        "after the groupby." % want)
            return "`by_group` has no %s column." % want
    for want in ("CONC_MEAN", "CONC_SD"):
        if want not in cols:
            return ("`by_group` has no %s column. Named aggregation, "
                    "CONC_MEAN=(\"CONC_NG_ML\", \"mean\"), gets the names right in one "
                    "step." % want)
    if len(by_group) != _EXP["by_group_rows"]:
        return ("`by_group` has %d rows and it should have %d, one per group and time."
                % (len(by_group), _EXP["by_group_rows"]))
    got = by_group.sort_values(["GROUP", "NOMINAL_H"])
    keys = ["%s|%g" % (g, t) for g, t in zip(got["GROUP"], got["NOMINAL_H"])]
    if keys != _EXP["by_group_keys"]:
        return "`by_group` does not hold one row per group and nominal time."
    if not _arr_close(got["CONC_MEAN"].to_numpy(), _EXP["by_group_mean"], rtol=1e-9):
        return "`CONC_MEAN` is not the mean concentration in each group at each time."
    arr = got["CONC_SD"].to_numpy()
    if _arr_close(arr, _EXP["by_group_sd_pop"], rtol=1e-6):
        return ("`CONC_SD` divides by n. Pandas .std() is the sample SD already, so this "
                "looks like ddof=0 was passed explicitly.")
    if not _arr_close(arr, _EXP["by_group_sd"], rtol=1e-9):
        return "`CONC_SD` is not the sample SD within each group and time."
    return None


def _part_b3():
    merged = _need("merged")
    n_missing_weight = _need("n_missing_weight")
    cols = _cols_of(merged)
    if cols is None:
        return "`merged` should be a pandas DataFrame."
    if "BODYWEIGHT_G" not in cols:
        return "`merged` has no BODYWEIGHT_G column, so the join did not bring demo in."
    if len(merged) < _EXP["merged_rows"]:
        return ("`merged` has %d rows and pk has %d. An inner join drops the animal with no "
                "demographics record, which hides exactly the thing you are looking for. "
                "Use how='left'." % (len(merged), _EXP["merged_rows"]))
    if len(merged) > _EXP["merged_rows"]:
        return ("`merged` has %d rows, more than the %d in pk, so a key was duplicated. "
                "validate='many_to_one' would have raised instead."
                % (len(merged), _EXP["merged_rows"]))

    got = _as_scalar(n_missing_weight)
    if got is None:
        return "`n_missing_weight` should be a single whole number."
    if int(got) == _EXP["n_missing_weight_rows"]:
        return ("`n_missing_weight` counted ROWS with no weight. The task asks for animals, "
                "so count distinct ANIMAL values with .nunique().")
    if int(got) != _EXP["n_missing_weight"]:
        return "`n_missing_weight` is not the number of animals with no body weight."
    return None


def _part_b4():
    frac = _need("frac_of_peak")
    arr = _as_array(frac)
    if arr is None:
        return "`frac_of_peak` should be a Series or array of numbers."
    if arr.size == _EXP["n_groups_animal"]:
        return ("`frac_of_peak` has one value per animal, so this is .agg or .max rather "
                "than .transform. A transform gives the group's value back on every one of "
                "its rows, which is what you need to divide by.")
    if arr.size != _EXP["frac_len"]:
        return ("`frac_of_peak` has %d values and pk has %d rows."
                % (arr.size, _EXP["frac_len"]))
    if int(_np.isfinite(arr).sum()) != _EXP["frac_notna"]:
        return ("`frac_of_peak` should still be nan wherever the concentration was nan, so "
                "%d real values out of %d." % (_EXP["frac_notna"], _EXP["frac_len"]))
    if not _num_close(float(_np.nansum(arr)), _EXP["frac_sum"], rtol=1e-9):
        return "`frac_of_peak` is not each concentration over that animal's own maximum."
    cols = _cols_of(globals().get("pk"))
    if cols is not None and "FRAC_OF_PEAK" not in cols:
        return "the column `FRAC_OF_PEAK` has not been added to pk."
    return None


def _part_b5():
    dev = _need("max_dev_min")
    wide = _need("wide")
    got = _as_scalar(dev)
    if got is None:
        return "`max_dev_min` should be a single number."
    if _num_close(got, _EXP["max_dev_h"], rtol=1e-6):
        return "`max_dev_min` is in hours. The task asks for minutes."
    if not _num_close(got, _EXP["max_dev_min"], rtol=1e-6):
        return ("`max_dev_min` is not the largest absolute deviation of ACTUAL_H from "
                "NOMINAL_H, in minutes.")

    cols = _cols_of(wide)
    if cols is None:
        return "`wide` should be a pandas DataFrame from pivot_table."
    if list(wide.shape) != _EXP["wide_shape"]:
        if list(wide.shape) == _EXP["wide_shape"][::-1]:
            return ("`wide` is the wrong way round. index=\"ANIMAL\" puts animals down the "
                    "rows and columns=\"NOMINAL_H\" puts times across.")
        return ("`wide` has shape %s and it should have shape %s. The pre dose column is "
                "entirely nan, and pivot_table drops it." % (list(wide.shape), _EXP["wide_shape"]))
    if not _arr_close([float(c) for c in wide.columns], _EXP["wide_cols"], rtol=1e-9):
        return "the columns of `wide` should be the nominal times."
    first = [_np.nan if v is None else v for v in _EXP["wide_first_row"]]
    if not _arr_close(wide.iloc[0].to_numpy(), first, rtol=1e-9):
        return "`wide` does not hold the concentrations for each animal at each time."
    return None


def _part_b6():
    inc = _need("inc")
    pct = _need("pct_high_event")
    if _cols_of(inc) is None:
        return "`inc` should be a table, from pd.crosstab."
    if list(inc.shape) != _EXP["inc_shape"]:
        return ("`inc` has shape %s and it should have shape %s, one row per group and one "
                "column per event value." % (list(inc.shape), _EXP["inc_shape"]))
    if [str(v) for v in inc.index] != _EXP["inc_index"]:
        return ("the rows of `inc` should be the dose groups, so GROUP is the first "
                "argument to crosstab.")
    if [str(v) for v in inc.columns] != _EXP["inc_cols"]:
        return "the columns of `inc` should be the EVENT values."
    if [[int(v) for v in row] for row in inc.to_numpy()] != _EXP["inc_values"]:
        return "`inc` does not hold the counts of animals in each group and event category."

    got = _as_scalar(pct)
    if got is None:
        return "`pct_high_event` should be a single number."
    if _num_close(got, _EXP["pct_high_event"] / 100.0, rtol=1e-6):
        return "`pct_high_event` is a fraction. The task asks for a percentage."
    if _num_close(got, _EXP["pct_high_of_study"], rtol=1e-6):
        return ("`pct_high_event` used the whole study as the denominator. Incidence is "
                "within group, so divide by the number of high dose animals.")
    if not _num_close(got, _EXP["pct_high_event"], rtol=1e-6):
        return "`pct_high_event` is not the percentage of high dose animals with the event."
    return None


def _part_c1():
    animal_level = _need("animal_level")
    group_summary = _need("group_summary")

    cols = _cols_of(animal_level)
    if cols is None:
        return "`animal_level` should be a pandas DataFrame."
    for want in ("ANIMAL", "GROUP", "MEAN_INHIB", "N_CELLS"):
        if want not in cols:
            idx = [str(n) for n in (animal_level.index.names or [])]
            if want in idx:
                return "`%s` is in the index rather than a column. Call .reset_index()." % want
            return "`animal_level` has no %s column." % want
    if len(animal_level) == _EXP["n_cells_total"]:
        return ("`animal_level` still has one row per cell. Group by animal before "
                "aggregating.")
    if len(animal_level) != _EXP["animal_rows"]:
        return ("`animal_level` has %d rows and there are %d animals."
                % (len(animal_level), _EXP["animal_rows"]))
    got = animal_level.sort_values("ANIMAL")
    if [int(v) for v in got["ANIMAL"]] != _EXP["animal_ids"]:
        return "`animal_level` does not list the eight animals."
    if [str(v) for v in got["GROUP"]] != _EXP["animal_groups"]:
        return ("GROUP does not line up with ANIMAL. Group by both at once so each animal "
                "keeps its own label.")
    if _arr_close(got["MEAN_INHIB"].to_numpy(), _EXP["animal_median"], rtol=1e-6):
        return "`MEAN_INHIB` is the median of each animal's cells, not the mean."
    if not _arr_close(got["MEAN_INHIB"].to_numpy(), _EXP["animal_mean"], rtol=1e-9):
        return "`MEAN_INHIB` is not the mean of each animal's own cells."
    if not _arr_close(got["N_CELLS"].to_numpy(), _EXP["animal_ncells"], rtol=1e-9):
        return "`N_CELLS` is not the number of cells in each animal."

    cols = _cols_of(group_summary)
    if cols is None:
        return "`group_summary` should be a pandas DataFrame."
    for want in ("GROUP", "MEAN", "SD", "N"):
        if want not in cols:
            idx = [str(n) for n in (group_summary.index.names or [])]
            if want in idx:
                return "`%s` is in the index rather than a column. Call .reset_index()." % want
            return "`group_summary` has no %s column." % want
    if len(group_summary) != len(_EXP["group_names"]):
        return ("`group_summary` has %d rows and there are %d groups."
                % (len(group_summary), len(_EXP["group_names"])))
    got = group_summary.sort_values("GROUP")
    if [str(v) for v in got["GROUP"]] != _EXP["group_names"]:
        return "`group_summary` does not hold one row per dose group."
    if _arr_close(got["N"].to_numpy(), _EXP["pooled_n"], rtol=1e-9):
        return ("`N` is the number of CELLS. The unit of replication is the animal, so N is "
                "the number of animals and `group_summary` has to be built from "
                "`animal_level`.")
    if not _arr_close(got["N"].to_numpy(), _EXP["group_n"], rtol=1e-9):
        return "`N` is not the number of animals in each group."
    if _arr_close(got["MEAN"].to_numpy(), _EXP["pooled_mean"], rtol=1e-6):
        return ("`MEAN` pools every cell, which weights each animal by however many cells it "
                "happened to have. Average the animal means instead.")
    if not _arr_close(got["MEAN"].to_numpy(), _EXP["group_mean"], rtol=1e-9):
        return "`MEAN` is not the mean of the animal means in each group."
    arr = got["SD"].to_numpy()
    if _arr_close(arr, _EXP["pooled_sd"], rtol=1e-6):
        return ("`SD` is the cell level spread, which is a statement about the assay. The "
                "SD you report is between animals, from the animal means.")
    if _arr_close(arr, _EXP["group_sem"], rtol=1e-6):
        return ("`SD` is the standard error. The house rule is SD, which does not shrink "
                "when you add animals.")
    if not _arr_close(arr, _EXP["group_sd"], rtol=1e-9):
        return "`SD` is not the sample SD of the animal means in each group."
    return None


def _part_c2():
    import matplotlib.figure as _mfig
    import matplotlib.axes as _maxes
    fig = _need("fig_c2")
    ax = _need("ax_c2")
    if not isinstance(fig, _mfig.Figure):
        return "`fig_c2` should be a matplotlib Figure, from plt.subplots()."
    if not isinstance(ax, _maxes.Axes):
        return "`ax_c2` should be a single matplotlib Axes."
    if not ax.containers and not ax.collections and not ax.lines:
        raise _Todo("the axes for c2 is still empty.")

    yerr = sorted(_yerr_values(ax))
    if not yerr:
        return ("no error bars were found on the axes. Draw the means with ax.errorbar and "
                "pass yerr, rather than plotting them as plain markers.")
    means = sorted(_plotted_means(ax))
    want_mean = sorted(float(v) for v in _EXP["group_mean"])
    if means and len(means) == len(want_mean):
        if _arr_close(means, sorted(float(v) for v in _EXP["pooled_mean"]), rtol=1e-3):
            return ("the plotted means come from pooling every cell rather than averaging "
                    "the animal means, so c1 needs fixing first.")
        if not _arr_close(means, want_mean, rtol=1e-3):
            return "the plotted means are not the group means from `group_summary`."
    want_sd = sorted(float(v) for v in _EXP["group_sd"])
    if len(yerr) != len(want_sd):
        return ("there are %d error bars on the axes and %d groups to plot."
                % (len(yerr), len(want_sd)))
    if _arr_close(yerr, sorted(float(v) for v in _EXP["group_sem"]), rtol=1e-2):
        return ("the error bars are the SEM. They should be the SD of the animal means, "
                "which is the house rule and is roughly twice as long here.")
    if _arr_close(yerr, sorted(float(v) for v in _EXP["pooled_sd"]), rtol=1e-2):
        return "the error bars are the cell level SD rather than the animal level SD."
    if not _arr_close(yerr, want_sd, rtol=1e-2):
        return "the error bar lengths do not match the SD of the animal means."
    if not ax.get_ylabel().strip():
        return "the y axis has no label. Say what is plotted and what the error bars are."
    if not ax.get_xlabel().strip():
        return "the x axis has no label."
    return None


def _part_c3():
    import matplotlib.figure as _mfig
    import matplotlib.axes as _maxes
    fig = _need("fig_c3")
    ax = _need("ax_c3")
    if not isinstance(fig, _mfig.Figure):
        return "`fig_c3` should be a matplotlib Figure, from plt.subplots()."
    if not isinstance(ax, _maxes.Axes):
        return "`ax_c3` should be a single matplotlib Axes."
    if not ax.lines and not ax.collections:
        raise _Todo("the axes for c3 is still empty.")

    if ax.get_yscale() != "log":
        return ("the y axis is still linear. ax.set_yscale(\"log\"), or ax.semilogy "
                "instead of ax.plot.")
    series = []
    for line in ax.lines:
        y = [float(v) for v in line.get_ydata()]
        if len(y) == len(_EXP["wide_cols"]):
            series.append(sorted(y))
    if len(series) < 2:
        return ("there should be one line per dose group, each with one point per nominal "
                "time, and %d were found." % len(series))
    want = [sorted(_EXP["by_group_mean"][:6]), sorted(_EXP["by_group_mean"][6:])]
    matched = 0
    for target in want:
        for got in series:
            if _arr_close(got, target, rtol=1e-3):
                matched += 1
                break
    if matched < 2:
        return ("the plotted lines are not the group mean concentrations from `by_group`. "
                "One line per group, sorted by nominal time.")
    if not ax.get_ylabel().strip() or not ax.get_xlabel().strip():
        return "both axes need labels."
    if ax.get_legend() is None:
        return "there is no legend, so nothing says which line is which dose."
    return None


def _part_d1():
    got = _need("q_sd")
    letter = str(got).strip().lower()[:1]
    if letter not in "abcd":
        return "`q_sd` should be one of the letters \"a\", \"b\", \"c\" or \"d\"."
    if letter == "a":
        return ("that is the spread of the individual cells, which describes the assay "
                "rather than the animals, and it uses the wrong unit of replication.")
    if letter == "c":
        return ("that is a standard error. It answers how precisely you located the mean, "
                "and it shrinks as you add animals whether or not the biology changes.")
    if letter == "d":
        return ("that is a standard error computed from cells, so it is wrong twice: the "
                "wrong level and the wrong quantity.")
    if letter != _EXP["q_sd"]:
        return "that is not the right option."
    return None


def _part_d2():
    got = _need("q_axis")
    letter = str(got).strip().lower()[:1]
    if letter not in "abcd":
        return "`q_axis` should be one of the letters \"a\", \"b\", \"c\" or \"d\"."
    if letter == "a":
        return ("that collapses the animals and leaves one number per sampling time, so six "
                "numbers rather than eight. The axis you name is the one that disappears.")
    if letter == "c":
        return "that reduces the whole matrix to a single number."
    if letter == "d":
        return ("that also reduces everything to one number, keeping the shape as (1, 1). "
                "keepdims preserves the dimensions, it does not choose an axis.")
    if letter != _EXP["q_axis"]:
        return "that is not the right option."
    return None


def _part_d3():
    got = _need("q_validate")
    letter = str(got).strip().lower()[:1]
    if letter not in "abcd":
        return "`q_validate` should be one of the letters \"a\", \"b\", \"c\" or \"d\"."
    if letter == "a":
        return ("that claims both sides are unique, which is false on the left: pk has seven "
                "rows per animal.")
    if letter == "b":
        return ("that is the right relationship stated backwards. The left frame is the one "
                "you call .merge on, and it is the many side here.")
    if letter == "d":
        return ("checking len() is a real habit and worth keeping, but it tells you only "
                "that something changed, after the fact. The question asks which argument "
                "states the expectation in the line that does the work.")
    if letter != _EXP["q_validate"]:
        return "that is not the right option."
    return None


check()


## The review

Run this cell after you have scored. It walks through every question with a model answer and the reasoning, including the ones you got right, several of which pass for a reason that is not obvious.

In [ ]:
review()


## Send to tutor

Run this and copy what it prints, along with any cell you want to talk about.

In [ ]:
print(send_to_tutor())
